<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/11_llm_assisted_causal_triplet_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install dependencies
!pip install torch transformers pandas accelerate

In [ ]:
import torch
import pandas as pd
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from tqdm import tqdm # For progress tracking

# 2. Paths
MODEL_PATH = "/content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/target_model_fp16"
INPUT_CSV = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set.csv"
OUTPUT_CSV = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set_traced.csv"

# 3. Load Model in 4-bit (Saves memory and massively speeds up text generation)
print("Loading Tokenizer and Model for Inference...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

# Using pipeline for easy inference management
generator = pipeline(
    "text-generation",
    model=MODEL_PATH,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
)

# 4. Define the Few-Shot Prompt Template for Phi-3
def build_extraction_prompt(article_text):
    """
    Forces Phi-3 to output strict JSON triplets using few-shot prompting.
    """
    sys_msg = "You are a precise data extraction AI. Extract the main subject and create a factual triplet for Causal Tracing. Output ONLY valid JSON, no other text."

    # Few-shot example 1
    user_1 = "Text: The former BBC Radio 1 DJ Tim Westwood has been interviewed by police under caution..."
    asst_1 = '{"clean_prompt": "The former BBC Radio 1 DJ is", "corrupted_prompt": "The famous American actor is", "target_token": " Westwood"}'

    # Few-shot example 2
    user_2 = "Text: Greek Prime Minister Kyriakos Mitsotakis has announced a new tax policy..."
    asst_2 = '{"clean_prompt": "The current Prime Minister of Greece is", "corrupted_prompt": "The current President of France is", "target_token": " Mitsotakis"}'

    # Actual target
    user_target = f"Text: {article_text[:600]}..." # Truncate to save context window

    prompt = f"<|system|>\n{sys_msg}<|end|>\n"
    prompt += f"<|user|>\n{user_1}<|end|>\n<|assistant|>\n{asst_1}<|end|>\n"
    prompt += f"<|user|>\n{user_2}<|end|>\n<|assistant|>\n{asst_2}<|end|>\n"
    prompt += f"<|user|>\n{user_target}<|end|>\n<|assistant|>\n"

    return prompt

# 5. Extraction Logic
def extract_json_from_response(response_text):
    """Safely extracts JSON from LLM output using Regex."""
    try:
        # Find anything that looks like JSON
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))
    except json.JSONDecodeError:
        pass
    return None

forget_df = pd.read_csv(INPUT_CSV)

# Fill NaNs with empty strings and force the column to string type.
forget_df['text'] = forget_df['text'].fillna("").astype(str)

results = []

print(f"\n--- Starting Triplet Extraction for {len(forget_df)} articles ---")

for index, row in tqdm(forget_df.iterrows(), total=len(forget_df)):
    article_text = row['text']

    if not article_text.strip():
        print(f"\n⚠️ Row {index} has no text. Skipping.")
        results.append({
            'id': index,
            'text': "",
            'clean_prompt': None,
            'corrupted_prompt': None,
            'target_token': None
        })
        continue

    prompt = build_extraction_prompt(article_text)

    # Generate response
    outputs = generator(
        prompt,
        max_new_tokens=120,
        max_length=None,
        temperature=0.1,
        return_full_text=False
    )

    response_text = outputs[0]['generated_text'].strip()
    triplet = extract_json_from_response(response_text)

    if triplet and all(k in triplet for k in ['clean_prompt', 'corrupted_prompt', 'target_token']):
        results.append({
            'id': index,
            'text': article_text,
            'clean_prompt': triplet['clean_prompt'],
            'corrupted_prompt': triplet['corrupted_prompt'],
            'target_token': triplet['target_token']
        })
    else:
        # Diagnostic Output
        print(f"\n⚠️ Failed to parse valid JSON for row {index}.")
        print(f"   Model Output was: {response_text}")
        results.append({
            'id': index,
            'text': article_text,
            'clean_prompt': None,
            'corrupted_prompt': None,
            'target_token': None
        })

# 7. Save and Clean Data
traced_df = pd.DataFrame(results)

# Drop rows where the LLM failed to generate valid JSON
success_rate = len(traced_df.dropna()) / len(traced_df) * 100
traced_df = traced_df.dropna()

traced_df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Triplet Extraction Complete! Success Rate: {success_rate:.1f}%")
print(f"✅ Saved to: {OUTPUT_CSV}")
print("\nPreview of extracted triplets:")
print(traced_df[['clean_prompt', 'target_token']].head(3))

Loading Tokenizer and Model for Inference...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]


--- Starting Triplet Extraction for 889 articles ---


 16%|█▌        | 143/889 [04:59<34:51,  2.80s/it][transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⚠️ Failed to parse valid JSON for row 142.
   Model Output was: {"clean_prompt": "The US Supreme Court is reviewing a decision by a federal judge in Texas that suspended approval by the Food and Drug Administration (FDA) of the abortion drug mifepristone, one of the most commonly used methods of terminating a pregnancy in America.", "corrupted_prompt": "The US Supreme Court is reviewing a decision by a federal judge in Texas that suspended approval by the Food and Drug Administration (FDA) of the cancer drug mifepristone, one of the most commonly


...


⚠️ Row 821 has no text. Skipping.


 94%|█████████▎| 832/889 [29:59<02:46,  2.91s/it][transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⚠️ Failed to parse valid JSON for row 831.
   Model Output was: {"clean_prompt": "The iconic fashion designer, who was born in the village of Tintwistle, Derbyshire, before moving to London, died in December. She was laid to rest in the village, where a florist - who had been tending to the grave at Westwood's family's request - was told of the theft.", "corrupted_prompt": "The famous British author, who was born in the city of Oxford, before moving to London, died in December. She was laid to rest in the city, where a flor


100%|██████████| 889/889 [32:09<00:00,  2.17s/it]


✅ Triplet Extraction Complete! Success Rate: 99.1%
✅ Saved to: /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set_traced.csv

Preview of extracted triplets:
                                        clean_prompt  target_token
0  The Greek Prime Minister has asked for forgive...    Mitsotakis
1                           The leader of the DUP is     Donaldson
2                             The iconic festival is   Glastonbury


In [11]:
import json
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

# Force Plotly to use the dedicated Google Colab rendering engine
pio.renderers.default = "colab"

# 1. Load the trace map
INPUT_JSON_PATH = "/content/drive/MyDrive/ResearchProject/trace_map.json"
# INPUT_JSON_PATH = "../data/processed/trace_map.json"
with open(INPUT_JSON_PATH, "r") as f:
    trace_map = json.load(f)

# 2. Prepare the dropdown options
dropdown_options = {}
for article_id, data in trace_map.items():
    snippet = data['text'][:50].replace('\n', ' ') + "..."
    label = f"Article {article_id} - {snippet}"
    dropdown_options[label] = article_id

# 3. Create a sandbox output area to prevent rendering blocks in Colab
output_area = widgets.Output()

# 4. Define the plotting function
def plot_saliency_map(selected_article_id):
    # Clear the previous visualization safely
    output_area.clear_output(wait=True)

    with output_area:
        # Extract data for the selected article
        data = trace_map[selected_article_id]
        scores = data['all_layer_scores']
        n_layers = len(scores)

        # Dynamic Sorting: Find the indices of the top 3 highest scores
        top_3_layers = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:3]

        # Create the color array (Red for the dynamic top 3 layers, else Blue)
        marker_colors = ['#ef4444' if idx in top_3_layers else '#3b82f6' for idx in range(n_layers)]

        # Generate the Plotly figure
        fig = go.Figure(data=[
            go.Bar(
                x=list(range(n_layers)),
                y=scores,
                marker_color=marker_colors,
                name="Recovery Score"
            )
        ])

        # Update layout with dynamic yaxis auto-scaling
        fig.update_layout(
            title=f"Mechanistic Localization for Article {selected_article_id}",
            xaxis_title="Transformer Layer",
            yaxis_title="Recovery of Target Fact (%)",
            yaxis=dict(autorange=True), # Enforces absolute auto-scaling for variable ranges
            template="plotly_white",
            height=500,
            margin=dict(l=40, r=40, t=60, b=40)
        )

        # Show context metadata above the graph
        print(f"Top 3 Identified Layers (Highest Impact): {top_3_layers}")
        print(f"Original Saved Layers: {data['top_layers']}\n")
        print(f"Target Text Snippet:\n{data['text'][:250]}...\n")

        # Render explicitly using colab protocol
        fig.show(renderer="colab")

# 5. Create the Interactive Widget
article_dropdown = widgets.Dropdown(
    options=dropdown_options,
    description='Select Data:',
    layout={'width': '80%'}
)

# 6. Event observer handler to securely update the canvas
def on_dropdown_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        plot_saliency_map(change['new'])

article_dropdown.observe(on_dropdown_change)

# Display the user interface components
print("--- Interactive Saliency Map Explorer ---")
display(article_dropdown, output_area)

# Initialize the notebook with the first plot pre-loaded
initial_key = list(dropdown_options.values())[0]
plot_saliency_map(initial_key)

--- Interactive Saliency Map Explorer ---


Dropdown(description='Select Data:', layout=Layout(width='80%'), options={'Article 0 - Greek Prime Minister Ky…

Output()